# Microsoft PowerPoint (2026년 최신 권장 사용법)

>[Microsoft PowerPoint](https://en.wikipedia.org/wiki/Microsoft_PowerPoint)는 Microsoft 에서 개발한 프레젠테이션 프로그램입니다.

이 노트북은 `Microsoft PowerPoint`(.pptx) 문서를 다운스트림에서 사용할 수 있는 `Document` 형식으로 로드하는 방법을 다룹니다.

> **⚠️ 2026년 9월 기준 변경 사항 (공통)**
>
> - 책에서 사용한 `langchain_community.document_loaders` 는 **`langchain-community` 패키지 sunset**(2026년 5월 발표, 6월 저장소 아카이브)으로 더 이상 유지보수되지 않습니다. 설치·import 는 여전히 되지만 새 프로젝트의 기반으로 권장되지 않습니다.
> - LangChain 의 현재 방향은 ① **전용 통합 패키지**(`langchain-upstage`, `langchain-unstructured`, `langchain-pymupdf4llm`, `langchain-docling` 등)를 쓰거나, ② 파싱 라이브러리를 **직접 사용**하고 결과를 `langchain_core.documents.Document` 로 감싸는 것(필요하면 `BaseLoader` 를 상속한 작은 로더 클래스 작성)입니다.
> - `Document`, `BaseLoader`, 텍스트 분할기(`langchain-text-splitters`)는 그대로 유지되는 핵심 인터페이스입니다.

**이 노트북에서 바뀐 점**

| 책(구버전, `langchain_community`) | 현재 권장 |
|---|---|
| `UnstructuredPowerPointLoader` | **`langchain-unstructured`** 의 `UnstructuredLoader` (요소 단위) |
| (신규) | **`python-pptx`** 직접 사용 — **슬라이드 단위** Document + 표·발표자 노트 포함 |

자세한 Unstructured 설정 방법은 [공식 도큐먼트](https://docs.unstructured.io/open-source/core-functionality/overview)를 참조하십시오.

In [ ]:
# 패키지 설치
# !pip install -qU langchain-core python-pptx
# !pip install -qU langchain-unstructured "unstructured[pptx]"

## UnstructuredLoader (구 `UnstructuredPowerPointLoader`)

`Unstructured` 는 텍스트의 다양한 **chunk** 에 대해 서로 다른 "요소(element)" 를 생성합니다.

책의 로더는 기본적으로 요소를 하나로 합쳐 반환했지만, `UnstructuredLoader` 는 **항상 요소 단위**로 반환합니다. (책의 `mode="elements"` 와 동일)

In [ ]:
from langchain_unstructured import UnstructuredLoader

# UnstructuredLoader 생성
loader = UnstructuredLoader("./data/sample-ppt.pptx")

# 데이터 로드
docs = loader.load()

# 로드한 요소(문서)의 개수 출력
print(len(docs))

In [ ]:
print(docs[0].page_content)

In [ ]:
docs[0].metadata

요소들을 **슬라이드(page_number) 별로 합치면** 슬라이드 단위 문서를 만들 수 있습니다.

In [ ]:
from collections import defaultdict

from langchain_core.documents import Document

by_slide: dict[int, list[str]] = defaultdict(list)
for d in docs:
    by_slide[d.metadata.get("page_number", 0)].append(d.page_content)

slide_docs = [
    Document(page_content="\n".join(texts), metadata={"source": "./data/sample-ppt.pptx", "slide": no})
    for no, texts in sorted(by_slide.items())
]
print(len(slide_docs))
print(slide_docs[0].page_content)

## python-pptx 로 직접 로드 (슬라이드 단위)

외부 파서 없이 `python-pptx` 만으로 슬라이드 제목, 본문 텍스트, 표, **발표자 노트**까지 추출할 수 있습니다.

In [ ]:
from pathlib import Path
from typing import Iterator

from langchain_core.document_loaders import BaseLoader
from pptx import Presentation


def shape_texts(shape) -> Iterator[str]:
    if getattr(shape, "has_text_frame", False) and shape.text_frame.text.strip():
        yield shape.text_frame.text.strip()
    if getattr(shape, "has_table", False):
        rows = [[c.text.strip() for c in row.cells] for row in shape.table.rows]
        yield "\n".join("| " + " | ".join(r) + " |" for r in rows)
    if hasattr(shape, "shapes"):  # 그룹 도형 내부 탐색
        for sub in shape.shapes:
            yield from shape_texts(sub)


class PptxSlideLoader(BaseLoader):
    """python-pptx 기반: 슬라이드 1장 = Document 1개"""

    def __init__(self, file_path: str | Path, include_notes: bool = True) -> None:
        self.file_path = Path(file_path)
        self.include_notes = include_notes

    def lazy_load(self) -> Iterator[Document]:
        prs = Presentation(self.file_path)
        for number, slide in enumerate(prs.slides, start=1):
            title_shape = slide.shapes.title
            title = title_shape.text_frame.text.strip() if title_shape is not None else ""
            texts = [t for shape in slide.shapes for t in shape_texts(shape)]
            if self.include_notes and slide.has_notes_slide:
                notes = slide.notes_slide.notes_text_frame.text.strip()
                if notes:
                    texts.append(f"[발표자 노트]\n{notes}")
            yield Document(
                page_content="\n\n".join(texts),
                metadata={"source": str(self.file_path), "slide": number, "title": title},
            )

In [ ]:
loader = PptxSlideLoader("./data/sample-ppt.pptx")
docs = loader.load()

print(len(docs))
print(docs[0].metadata)
print(docs[0].page_content)